# Who gains most from a coupon: an effect that changes with age

## At a glance

A retailer sent a discount coupon to some of its customers and wants to know which customers it
should send coupons to next time. The question is how the coupon's effect on the chance of a
purchase changes with the customer's age.

The estimated effect falls from about 24 percentage points for 20-year-olds to about 3 points for
70-year-olds, close to the true curve of the simulated data. Sending coupons only where the
estimated gain beats the coupon's cost earns 0.0523 net purchases per customer, against 0.0546
for the best possible rule and 0.0293 for a coupon to everyone. The main caveat: the curve has no
confidence interval, so check it on held-out rows, as this page does, before you act on it.

## The question

The marketing team chose who got a coupon, and it sent more coupons to customers who spent more
last year. Those customers also buy more anyway. The question is, for customers of a given age,
how much a coupon raises the chance that they buy in the next month, compared with no coupon.
This is the conditional average treatment effect (CATE) given age: the average effect among
customers of that age.

A second, shorter analysis asks the same kind of question of a randomized drug trial: does the
drug's effect on the risk of an event change with how severe the patient's illness is?

## The data

Every fit sets its seed, so a rerun prints the same numbers. Another version of `cleverly` can
print slightly different numbers.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

import cleverly

pd.set_option("display.width", 130)
print("cleverly", cleverly.__version__)

cleverly 0.2.0


The page simulates 2,000 customers, so it knows the true effect at every age.

In [2]:
def true_uplift(age):
    """The coupon's true effect on the chance of a purchase, by age."""
    return 0.25 / (1.0 + np.exp((age - 45.0) / 8.0))


def simulate_coupons(n, seed):
    rng = np.random.default_rng(seed)
    age = rng.uniform(18, 80, n)
    spend = rng.uniform(0, 1, n)  # last year's spend, scaled to 0-1
    visits = rng.poisson(3, n).clip(0, 10)  # store visits last month
    # The marketing team sent more coupons to big spenders.
    chance = 1.0 / (1.0 + np.exp(-(-1.0 + 2.5 * spend + 0.02 * (age - 50))))
    coupon = rng.binomial(1, chance)
    buys = 0.15 + 0.4 * spend + 0.02 * visits + coupon * true_uplift(age)
    bought = rng.binomial(1, buys)
    return pd.DataFrame(
        {"age": age, "spend": spend, "visits": visits, "coupon": coupon, "bought": bought}
    )


# With your own data, load the data frame here, such as frame = pd.read_csv(...), and drop
# every line that uses true_uplift: real data have no known answer.
frame = simulate_coupons(n=2_000, seed=1)
print("rows and columns:", frame.shape)
print("column names:", list(frame.columns))
print(frame.head().round(3).to_string())

rows and columns: (2000, 5)
column names: ['age', 'spend', 'visits', 'coupon', 'bought']
      age  spend  visits  coupon  bought
0  49.733  0.284       5       0       1
1  76.929  0.010       4       0       0
2  26.938  0.110       4       0       1
3  76.816  0.131       2       0       0
4  37.334  0.432       2       1       1


Each row is one customer: age, last year's spend (scaled from 0 to 1), store visits last
month, whether they got a coupon, and whether they bought in the following month. The true
coupon effect falls smoothly with age, from about 24 percentage points for the youngest customers
to under 2 points after 65, and it does not depend on spend or visits.

## Why a simple comparison misleads

In [3]:
sent = frame["coupon"] == 1
print(f"share sent a coupon:          {sent.mean():.3f}")
print(
    f"mean spend, sent / not sent:  {frame.loc[sent, 'spend'].mean():.3f} / "
    f"{frame.loc[~sent, 'spend'].mean():.3f}"
)
print()
bands = pd.cut(frame["age"], [18, 35, 50, 65, 80], include_lowest=True)
naive = frame.groupby([bands, "coupon"], observed=True)["bought"].mean().unstack()
table = pd.DataFrame(
    {
        "simple difference": naive[1] - naive[0],
        "true effect": frame.groupby(bands, observed=True)["age"].apply(
            lambda ages: true_uplift(ages).mean()
        ),
    }
)
print(table.round(3).to_string())

share sent a coupon:          0.547
mean spend, sent / not sent:  0.579 / 0.390

                simple difference  true effect
age                                           
(17.999, 35.0]              0.328        0.224
(35.0, 50.0]                0.232        0.143
(50.0, 65.0]                0.126        0.047
(65.0, 80.0]                0.120        0.009


Customers who got a coupon spent more last year (0.579 against 0.390 on the scaled spend), and
big spenders buy more with or without a coupon. So within each age band, the simple difference
in purchase rates between customers with and without a coupon overstates the effect: 0.328
against a true 0.224 for the youngest band, and 0.120 against 0.009 for the oldest. Targeting
on these numbers would send coupons to older customers who gain almost nothing.

## Set up the analysis

Tell `cleverly` which column is the treatment, which is the outcome, and which to adjust for, and
name the column the effect may change with (`Modifiers`). The domain is the range of ages the
curve covers.

In [4]:
from cleverly import CausalStudy, ConditionalContrast, PointTreatment
from cleverly.bases import Modifiers

# For your own data, change the column names. `domains` gives the range the curve covers.
study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="bought", treatment="coupon", adjustment=("age", "spend", "visits")
    ),
)
by_age = Modifiers(["age"], domains={"age": (18, 80)})
effect = study.identify(ConditionalContrast(by_age))
print("\n".join(effect.summary().splitlines()[:37]))

                            Causal question (cate)
Question:      conditional average      Rows:                            2,000
               treatment effect given   Answered by:  adjusting for covariates
               age
Outcome:       bought (binary)
Treatment:     coupon (levels 0, 1)
Adjusted for:  age, spend, visits
Conditions the answer rests on
condition                checkable   what it says
------------------------------------------------------------------------------
well-defined treatment   no          Each unit's observed outcome is the
                                     outcome it would have had under the
                                     treatment it actually received. This
                                     needs the treatment to be well defined:
                                     every way of getting the same level has
                                     the same effect.
no interference          no          One unit's treatment does not change
         

The summary lists three conditions the answer needs, none of which the data can check. "No
hidden confounding" asks that, among customers of the same age, spend and visits, nothing else
drove both the coupon and the purchase; here the simulation makes it true, but with your data it
needs judgment about how the team chose. The full summary adds overlap: every kind of customer
had a real chance of getting a coupon and of not getting one.

## Estimate how the effect changes with age

The method is the efficient plug-in (EP) learner of van der Laan, Carone and Luedtke. It fits an
outcome model (who buys, from the coupon and the customer's columns) and a treatment model (who
got a coupon), each row predicted by models trained without it, like out-of-fold predictions in
cross-validation. It then corrects the outcome model with one weighted regression on a few
smooth functions of age, and fits the curve to the corrected effect. Cross-validation chooses how
many functions of age to use.

In [5]:
# For your own data, swap in your own learners. Out-of-fold predictions keep a flexible
# learner honest, so a random forest or gradient boosting works here too.
curve = effect.estimate(
    "ep",
    outcome_learner=LogisticRegression(max_iter=1000),
    treatment_learner=LogisticRegression(max_iter=1000),
    random_state=0,
)
ages = np.array([20, 30, 40, 50, 60, 70, 80])
print(
    pd.DataFrame(
        {
            "age": ages,
            "estimated effect": curve.predict({"age": ages}),
            "true effect": true_uplift(ages),
        }
    )
    .round(3)
    .to_string(index=False)
)
print()
print("functions of age the curve uses:", curve.sieve.chosen)
print("label:", curve.status)

 age  estimated effect  true effect
  20             0.239        0.239
  30             0.220        0.217
  40             0.177        0.163
  50             0.122        0.087
  60             0.069        0.033
  70             0.030        0.011
  80             0.016        0.003

functions of age the curve uses: 1
label: experimental


The estimated curve follows the truth: 0.239 at age 20, where the truth is 0.239, and 0.030 at
age 70 against 0.011. It is a little high between 50 and 70. Cross-validation chose one function
of age, the smoothest shape on offer, which suits a curve that only falls.

Both models here are logistic regressions on the raw columns. The outcome model gives the coupon
one term for every customer, so it misses how the effect changes with age, but the treatment
model is right, and the correction step uses it to recover the curve. The curve has no confidence
interval, because no theory yet gives one for a curve like this. `cleverly` marks the fit
experimental: the method's theory does not cover a number of functions chosen by
cross-validation, and no simulation study has yet checked the method.

## Check the curve on held-out rows

Without an interval, check the curve on rows it did not see. Each check refits the whole analysis
without one fold of rows and compares the curve's predictions on that fold with a corrected
estimate of each customer's effect.

In [6]:
calibration = curve.diagnostics.calibration()
print(f"calibration line: intercept {calibration.intercept:.3f}, slope {calibration.slope:.2f}")
print()
risks = curve.diagnostics.risk_comparison().risks
print("out-of-fold error (lower is better):")
for name in ("curve", "constant", "t_learner"):
    print(f"  {name:10s} {risks[name]:.4f}")

calibration line: intercept 0.028, slope 0.76



out-of-fold error (lower is better):
  curve      -0.0197
  constant   -0.0147
  t_learner  -0.0167


The calibration line regresses those corrected estimates on the held-out predictions. A slope of
1 means the curve's differences between ages are right on average; this slope of 0.76 says they
are somewhat too large, so read the curve's ranking of ages more firmly than its exact values.
The out-of-fold error compares the curve with two simpler answers: one effect for every age
(`constant`), and the effect read straight off the outcome model (`t_learner`). The curve has the lowest error (-0.0197, against
-0.0147 and -0.0167), so knowing age improves the prediction. The error is negative because it
leaves out a term that is the same for every candidate.

## Decide who gets a coupon

A coupon costs the retailer the profit of 0.08 purchases. The rule below sends a coupon to each
customer whose estimated gain is larger than that cost.

In [7]:
# A coupon costs the retailer the profit of 0.08 purchases. Send it where the
# estimated gain is larger.
COST = 0.08
print(f"cost of one coupon, in purchases:  {COST:.2f}")
target = curve.predict({"age": frame["age"]}) > COST
gain = true_uplift(frame["age"]) - COST
print(f"share of customers targeted:      {target.mean():.3f}")
print(
    f"targeted ages:                    {frame.loc[target, 'age'].min():.0f} to "
    f"{frame.loc[target, 'age'].max():.0f}"
)
print(f"true net gain per customer, rule: {np.mean(gain * target):.4f}")
print(f"true net gain, best possible:     {np.mean(gain * (gain > 0)):.4f}")
print(f"true net gain, coupon for all:    {np.mean(gain):.4f}")

cost of one coupon, in purchases:  0.08
share of customers targeted:      0.643
targeted ages:                    18 to 58
true net gain per customer, rule: 0.0523
true net gain, best possible:     0.0546
true net gain, coupon for all:    0.0293


The rule sends coupons to customers aged 18 to 58, 64% of them. Because the data are simulated,
the page can score the rule with the true effect: it earns 0.0523 net purchases per customer,
close to the 0.0546 of the best possible rule, and well above the 0.0293 of a coupon for
everyone. The rule stops a little late, because the curve runs high in the fifties.

## A trial: does the drug help sicker patients more?

The same method estimates a ratio. In a simulated trial of 2,000 patients, the drug was assigned
at random, and the question is how the risk ratio (the risk of an event with the drug divided by
the risk without it) changes with a severity score from 0 to 1. For a sample this small, the code
fixes the curve to one smooth function of severity before the fit (`size=1`), rather than
letting cross-validation choose.

In [8]:
from cleverly.bases import Cosine, Sieve


def simulate_trial(n, seed):
    rng = np.random.default_rng(seed)
    severity = rng.uniform(0, 1, n)  # a baseline severity score, 0-1
    age = rng.uniform(40, 80, n)
    drug = rng.binomial(1, 0.5, n)  # assigned at random
    risk = 0.20 + 0.40 * severity + 0.002 * (age - 60)
    event = rng.binomial(1, risk * np.exp(drug * (-0.8 * severity)))
    return pd.DataFrame({"severity": severity, "age": age, "drug": drug, "event": event})


trial = simulate_trial(n=2_000, seed=1)
trial_effect = CausalStudy(
    trial,
    design=PointTreatment(outcome="event", treatment="drug", adjustment=("severity", "age")),
).identify(
    ConditionalContrast(Modifiers(["severity"], domains={"severity": (0, 1)}), scale="log_ratio")
)
ratio_curve = trial_effect.estimate(
    "ep",
    outcome_learner=LogisticRegression(max_iter=1000),
    treatment_learner=LogisticRegression(max_iter=1000),
    sieve=Sieve(Cosine(["severity"], 6), size=1),
    random_state=0,
)
levels = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
print(
    pd.DataFrame(
        {
            "severity": levels,
            "estimated risk ratio": np.exp(ratio_curve.predict({"severity": levels})),
            "true risk ratio": np.exp(-0.8 * levels),
        }
    )
    .round(2)
    .to_string(index=False)
)
trial_calibration = ratio_curve.diagnostics.calibration()
print()
print(f"calibration slope: {trial_calibration.slope:.2f}")

 severity  estimated risk ratio  true risk ratio
     0.00                  1.03             1.00
     0.25                  0.92             0.82
     0.50                  0.70             0.67
     0.75                  0.54             0.55
     1.00                  0.48             0.45



calibration slope: 0.89


The estimated risk ratio falls from 1.03 for the least severe patients, where the drug does
nothing, to 0.48 for the most severe, close to the true 0.45. The calibration slope of 0.89 is
near 1. In this trial every patient had the same chance of the drug, so the treatment model is
right, which is what the correction step needs.

## What we can and can't conclude

For the marketing team: the coupon's effect falls with age, and sending it only to customers
under about 58 earns almost as much as the best possible rule and nearly twice as much as a
coupon for everyone. That rests on two things the data cannot check: that age, spend and visits
account for how the team chose who got a coupon, and that every kind of customer had a real
chance of either option. The curve has no interval, so the held-out checks are the main evidence
that it is not noise.

The page does not show that the estimated curve is the true curve for your data, or that a rule
learned on one sample of customers earns the same on the next. It shows one simulated sample with
a known answer, which does not show how the method performs in general.

## Go deeper

### How we know the method works

No simulation study has yet checked this method in `cleverly`. The `ep-cate`, `ep-crr` and
`ep-assessment` studies are declared and have not run; the
[conditional effects reference](../technical-reference/conditional-effects.md) states what they
will measure and what the theory claims. The package's tests check the correction step exactly on
small made-up populations whose answer is known, including when the outcome model is wrong.

### Does the curve depend on how the rows were split?

The out-of-fold fits split the rows at random. `stability()` refits the curve under five
different splits of the same rows and compares the spread of those refits with the spread of the curve
across ages.

In [9]:
stability = curve.diagnostics.stability()
print(f"spread across 5 refits, relative to the curve's own spread: {stability.relative_sd:.3f}")
print("warning:", stability.warning)

spread across 5 refits, relative to the curve's own spread: 0.208


The refits move by about a fifth (0.208) of the curve's own spread, well below the level at
which `cleverly` warns (a half). This measures sensitivity to the split on the same data, not
sampling noise, so a low value does not show that the curve is well determined.

## Where to go next

The [conditional effects guide](../user-guide/conditional-effects.md) shows how to declare a
simple shape for the effect and get intervals for it. The
[one treatment, one outcome](point-treatment-tmle.ipynb) example explains the outcome and
treatment models step by step, and [who should get the offer](treatment-rules.ipynb) learns a
treatment rule directly. The [examples index](index.md) lists every example.